# Household-day construction and leakage-free target

This notebook implements Stages 0–3 of the modelling-data pipeline for the MSc research project **Calibrated and Explainable Machine Learning for Day-Ahead Household High-Demand Risk Prediction Using Sri Lankan Smart Meter Data**.

The prediction unit is one household on one local calendar day. The notebook:

1. records the scientific configuration and input contracts;
2. validates, but never modifies, the completed LP interval dataset;
3. creates a continuous household-day panel with observed and complete-day metrics; and
4. constructs household-relative P95 and P90 targets from the preceding 45 calendar days only.

The notebook reads the frozen primary policy from `03a_target_design_experiments.ipynb` and verifies that its target-day, history-day, window, support, quantile, and quantile-method settings match the saved experiment evidence. A mismatch or non-frozen status stops the run. The resulting target labels use that exact configuration.

It does **not** engineer model predictors or train a model. High interval values are retained. Missing demand, target days below the selected coverage threshold, and insufficient history are never imputed into the target. It exports observed household-days separately from the continuous calendar panel used for rolling calculations.

## Stage 0 — Configuration and reproducibility

In [ ]:
from pathlib import Path
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from collections import Counter
import hashlib
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.6f}")
sns.set_theme(style="whitegrid", context="notebook")

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "data").exists() else CURRENT_DIR.parent
if not (PROJECT_ROOT / "data").is_dir() or not (PROJECT_ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Could not identify project root: {PROJECT_ROOT}")

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLE_DIR = REPORTS_DIR / "tables" / "03_household_day_target"
FIGURE_DIR = REPORTS_DIR / "figures" / "03_household_day_target"

INTERVAL_PATH = PROCESSED_DIR / "01_preprocessed_lp_interval_data.csv"
INTERVAL_DTYPE_CONTRACT_PATH = PROCESSED_DIR / "01_preprocessing_dtype_contract.json"
COVERAGE_PATH = INTERIM_DIR / "01_household_day_coverage_audit.csv"
PREPROCESSING_MANIFEST_PATH = PROCESSED_DIR / "01_preprocessing_manifest.json"
EDA_MANIFEST_PATH = REPORTS_DIR / "tables" / "02_interval_eda" / "02_eda_manifest.json"
TARGET_DESIGN_RESULTS_PATH = REPORTS_DIR / "tables" / "03a_target_design_experiments" / "03a_target_design_results.csv"
TARGET_MASKING_RESULTS_PATH = REPORTS_DIR / "tables" / "03a_target_design_experiments" / "03a_completeness_masking_results.csv"
HISTORY_POLICY_RESULTS_PATH = REPORTS_DIR / "tables" / "03a_target_design_experiments" / "03a_history_quality_support_experiment.csv"
TARGET_CONFIG_PATH = REPORTS_DIR / "tables" / "03a_target_design_experiments" / "03a_selected_target_config.json"
DAILY_OUTPUT_PATH = PROCESSED_DIR / "02_daily_household_data.csv"
TARGET_OUTPUT_PATH = PROCESSED_DIR / "03_daily_with_target.csv"
OBSERVED_DAILY_OUTPUT_PATH = PROCESSED_DIR / "02_observed_household_day.csv"
OBSERVED_DAILY_DTYPE_CONTRACT_PATH = PROCESSED_DIR / "02_observed_household_day_dtype_contract.json"
DAILY_DTYPE_CONTRACT_PATH = PROCESSED_DIR / "02_daily_household_data_dtype_contract.json"
TARGET_DTYPE_CONTRACT_PATH = PROCESSED_DIR / "03_daily_with_target_dtype_contract.json"
CONFIG_OUTPUT_PATH = TABLE_DIR / "03_configuration.json"
RUN_MANIFEST_PATH = TABLE_DIR / "03_run_manifest.json"

CSV_ENCODING = "utf-8"
CSV_SEPARATOR = ","
CSV_MISSING_VALUE = ""
CSV_CHUNK_SIZE = 250_000
PUBLISHED_TABLE_PATHS = []

for directory in [PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

required_inputs = [INTERVAL_PATH, INTERVAL_DTYPE_CONTRACT_PATH, COVERAGE_PATH, PREPROCESSING_MANIFEST_PATH, EDA_MANIFEST_PATH, TARGET_DESIGN_RESULTS_PATH, TARGET_MASKING_RESULTS_PATH, HISTORY_POLICY_RESULTS_PATH, TARGET_CONFIG_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json_atomic(payload, path):
    temp_path = path.with_suffix(path.suffix + ".tmp")
    temp_path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    temp_path.replace(path)


def file_sha256(path, block_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def csv_signature(path, rows, columns, sha256=None):
    return {
        "path": str(path), "format": "csv", "size_bytes": int(path.stat().st_size),
        "modified_ns": int(path.stat().st_mtime_ns), "rows": int(rows),
        "columns": int(columns), "sha256": sha256 or file_sha256(path),
    }


def input_signature(path, manifest):
    signature = manifest["primary_signature"]
    return {
        "path": str(path), "format": "csv", "size_bytes": int(path.stat().st_size),
        "modified_ns": int(path.stat().st_mtime_ns), "rows": int(manifest["primary_rows"]),
        "columns": len(manifest["primary_columns"]), "manifest_sha256": signature["sha256"],
    }


def write_csv_atomic(frame, path):
    temporary_path = path.with_suffix(".tmp.csv")
    if temporary_path.exists():
        temporary_path.unlink()
    frame.to_csv(
        temporary_path, index=False, encoding=CSV_ENCODING, sep=CSV_SEPARATOR,
        na_rep=CSV_MISSING_VALUE, date_format="%Y-%m-%d %H:%M:%S", float_format="%.17g",
    )
    temporary_path.replace(path)


def logical_dtype(dtype):
    if pd.api.types.is_datetime64_any_dtype(dtype):
        return "datetime"
    if pd.api.types.is_bool_dtype(dtype):
        return "boolean"
    if pd.api.types.is_integer_dtype(dtype):
        return str(dtype)
    if pd.api.types.is_float_dtype(dtype):
        return str(dtype)
    return "string"


def write_dtype_contract(frame, dataset_path, contract_path):
    column_types = {column: logical_dtype(frame[column].dtype) for column in frame.columns}
    payload = {
        "contract_version": 1, "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "path": str(dataset_path), "encoding": CSV_ENCODING, "separator": CSV_SEPARATOR,
        "missing_value": CSV_MISSING_VALUE,
        "date_columns": [column for column, dtype in column_types.items() if dtype == "datetime"],
        "columns": column_types,
        "reader": {"true_values": ["True"], "false_values": ["False"], "keep_default_na": False, "na_values": [""]},
    }
    write_json_atomic(payload, contract_path)
    return payload


def read_dataset_csv(path, contract, usecols=None, chunksize=None):
    selected = list(contract["columns"]) if usecols is None else list(usecols)
    date_columns = [column for column in contract.get("date_columns", []) if column in selected]
    dtype_map = {
        column: dtype for column, dtype in contract["columns"].items()
        if column in selected and dtype != "datetime"
    }
    return pd.read_csv(
        path, usecols=selected, dtype=dtype_map or None, parse_dates=date_columns or None,
        true_values=contract["reader"]["true_values"], false_values=contract["reader"]["false_values"],
        keep_default_na=contract["reader"]["keep_default_na"], na_values=contract["reader"]["na_values"],
        encoding=contract["encoding"], sep=contract["separator"], chunksize=chunksize, low_memory=False,
    )


def publish_table(frame, filename, title=None, display_rows=20):
    path = TABLE_DIR / filename
    if path.suffix.lower() != ".csv":
        raise ValueError(f"Report tables must be CSV: {path}")
    write_csv_atomic(frame, path)
    PUBLISHED_TABLE_PATHS.append(path)
    if title:
        print(f"{title} — saved to {path.name}")
    display(frame.head(display_rows))
    return path


def publish_figure(fig, filename):
    path = FIGURE_DIR / filename
    fig.savefig(path, dpi=160, bbox_inches="tight", facecolor="white")
    plt.show()
    plt.close(fig)
    print(f"Saved figure: {path.name}")
    return path


preprocessing_manifest = read_json(PREPROCESSING_MANIFEST_PATH)
interval_dtype_contract = read_json(INTERVAL_DTYPE_CONTRACT_PATH)
eda_manifest = read_json(EDA_MANIFEST_PATH)
target_design_results = pd.read_csv(TARGET_DESIGN_RESULTS_PATH, low_memory=False)
target_masking_results = pd.read_csv(TARGET_MASKING_RESULTS_PATH, low_memory=False)
history_policy_results = pd.read_csv(HISTORY_POLICY_RESULTS_PATH, low_memory=False)
selected_target_config = read_json(TARGET_CONFIG_PATH)
if selected_target_config.get("status") != "frozen_primary_research_policy":
    raise RuntimeError("03A target configuration is not frozen; refusing to build downstream target labels.")
selected_parameters = selected_target_config["selected_parameters"]
selected_target_min = int(selected_parameters["target_min_intervals"])
selected_history_min = int(selected_parameters["history_min_intervals"])
selected_window_days = int(selected_parameters["historical_window_days"])
selected_history_fraction = float(selected_parameters["minimum_history_fraction"])
selected_min_history_days = int(selected_parameters["min_valid_history_days"])
selected_quantile = float(selected_parameters["primary_quantile"])
selected_quantile_method = str(selected_parameters["quantile_method"])
selected_window_closed = str(selected_parameters["rolling_window_closed"])

target_design_candidate = target_design_results.loc[
    target_design_results["target_min_intervals"].eq(selected_target_min)
    & target_design_results["history_min_intervals"].eq(selected_history_min)
    & np.isclose(target_design_results["minimum_history_fraction"], selected_history_fraction)
    & target_design_results["historical_window_days"].eq(selected_window_days)
].copy()
masking_candidate = target_masking_results.loc[
    target_masking_results["row_type"].eq("summary")
    & target_masking_results["target_min_intervals"].eq(selected_target_min)
].copy()
history_policy_candidate = history_policy_results.loc[
    history_policy_results["target_min_intervals"].eq(selected_target_min)
    & history_policy_results["history_min_intervals"].eq(selected_history_min)
    & np.isclose(history_policy_results["minimum_history_fraction"], selected_history_fraction)
    & history_policy_results["historical_window_days"].eq(selected_window_days)
].copy()
if len(target_design_candidate) != 1 or len(masking_candidate) != 1 or len(history_policy_candidate) != 1:
    raise RuntimeError("Frozen 03A target configuration does not match exactly one saved experiment row.")
policy_evidence = selected_target_config.get("decision_evidence", {})
if not np.isclose(float(masking_candidate.iloc[0]["label_agreement_rate"]), float(policy_evidence["target_day_94_label_agreement"]), atol=1e-12):
    raise RuntimeError("Frozen target config label-agreement evidence differs from the saved masking experiment.")
if int(history_policy_candidate.iloc[0]["eligible_household_days"]) != int(policy_evidence["history_92_75_eligible_household_days"]):
    raise RuntimeError("Frozen target config population evidence differs from the saved history experiment.")
previous_run_manifest = read_json(RUN_MANIFEST_PATH) if RUN_MANIFEST_PATH.exists() else None
input_signature_before = input_signature(INTERVAL_PATH, preprocessing_manifest)

INTERVAL_COLUMNS = list(interval_dtype_contract["columns"])
INTERVAL_DATE_COLUMNS = set(interval_dtype_contract["date_columns"])
INTERVAL_READ_DTYPES = {name: dtype for name, dtype in interval_dtype_contract["columns"].items() if dtype != "datetime"}


def iter_interval_chunks(columns=None, chunksize=CSV_CHUNK_SIZE):
    selected = INTERVAL_COLUMNS if columns is None else list(columns)
    unknown = sorted(set(selected) - set(INTERVAL_COLUMNS))
    if unknown:
        raise KeyError(f"Unknown interval columns: {unknown}")
    date_columns = [name for name in selected if name in INTERVAL_DATE_COLUMNS]
    dtype_map = {name: INTERVAL_READ_DTYPES[name] for name in selected if name in INTERVAL_READ_DTYPES}
    yield from pd.read_csv(
        INTERVAL_PATH, usecols=selected, dtype=dtype_map, parse_dates=date_columns or None,
        true_values=["True"], false_values=["False"], keep_default_na=False, na_values=[""],
        encoding=CSV_ENCODING, chunksize=chunksize, low_memory=False,
    )


def iter_complete_household_chunks(columns, chunksize=CSV_CHUNK_SIZE):
    selected = list(dict.fromkeys(["household_id", *columns]))
    carry = None
    for chunk in iter_interval_chunks(selected, chunksize=chunksize):
        if carry is not None:
            chunk = pd.concat([carry, chunk], ignore_index=True)
        if chunk.empty:
            continue
        last_household = chunk["household_id"].iloc[-1]
        final_group = chunk["household_id"].eq(last_household)
        complete = chunk.loc[~final_group]
        carry = chunk.loc[final_group].copy()
        if not complete.empty:
            yield complete
    if carry is not None and not carry.empty:
        yield carry


print(f"Project root: {PROJECT_ROOT}")
print(f"Interval input: {INTERVAL_PATH}")
print(f"Interval rows from validated manifest: {input_signature_before['rows']:,}")
print(f"Frozen target policy: {selected_target_config['configuration_id']} ({selected_target_min}/96 target, {selected_history_min}/96 history, {selected_window_days} days, {selected_min_history_days} supported days)")
print(f"CSV chunk size: {CSV_CHUNK_SIZE:,}")


In [ ]:
@dataclass(frozen=True)
class HouseholdDayTargetConfig:
    timezone: str
    interval_minutes: int
    expected_intervals_per_day: int
    near_complete_min_intervals: int
    target_day_min_intervals: int
    history_day_min_intervals: int
    target_history_window_days: int
    minimum_history_fraction: float
    target_min_history_days: int
    sensitivity_history_days: tuple
    primary_target_percentile: float
    sensitivity_target_percentile: float
    target_quantile_method: str
    rolling_window_closed: str
    target_configuration_id: str
    random_state: int = 42
    execution_mode: str = "rebuild"

selected_policy = selected_target_config["selected_parameters"]
preprocessing_policy = preprocessing_manifest["policy"]
window_days = int(selected_policy["historical_window_days"])
config = HouseholdDayTargetConfig(
    timezone=str(preprocessing_policy["timezone"]),
    interval_minutes=int(preprocessing_policy.get("interval_minutes", 15)),
    expected_intervals_per_day=int(preprocessing_policy["expected_intervals_per_day"]),
    near_complete_min_intervals=70,
    target_day_min_intervals=int(selected_policy["target_min_intervals"]),
    history_day_min_intervals=int(selected_policy["history_min_intervals"]),
    target_history_window_days=window_days,
    minimum_history_fraction=float(selected_policy["minimum_history_fraction"]),
    target_min_history_days=int(selected_policy["min_valid_history_days"]),
    sensitivity_history_days=(int(np.ceil(window_days * .70)), int(np.ceil(window_days * .80))),
    primary_target_percentile=float(selected_policy["primary_quantile"]),
    sensitivity_target_percentile=.90,
    target_quantile_method=str(selected_policy["quantile_method"]),
    rolling_window_closed=str(selected_policy["rolling_window_closed"]),
    target_configuration_id=str(selected_target_config["configuration_id"]),
)
INTERVAL_HOURS = config.interval_minutes / 60.0

evidence = selected_target_config.get("decision_evidence", {})
evidence_table = pd.DataFrame([
    ("Configuration ID", config.target_configuration_id, "Loaded from the frozen 03A target-design decision."),
    ("Selection status", selected_target_config["status"], "A non-frozen 03A configuration stops this notebook."),
    ("Canonical timezone", config.timezone, "Inherited from the validated preprocessing policy."),
    ("Interval duration", config.interval_minutes, "Observed LP cadence in minutes."),
    ("Interval hours", INTERVAL_HOURS, "Used only for interval-energy conversion."),
    ("Expected intervals per day", config.expected_intervals_per_day, "Inherited from preprocessing policy."),
    ("Target-day requirement", config.target_day_min_intervals, f"Frozen 03A policy; label agreement={float(evidence['target_day_94_label_agreement']):.4%}."),
    ("History-day requirement", config.history_day_min_intervals, "Frozen 03A policy for contributing historical days."),
    ("Historical window", config.target_history_window_days, f"Calendar days; rolling window closed={config.rolling_window_closed}."),
    ("Minimum valid history", config.target_min_history_days, f"Frozen support rule: ceil({config.minimum_history_fraction:.0%} × {config.target_history_window_days})."),
    ("Primary percentile", config.primary_target_percentile, "Frozen household-relative primary target."),
    ("Sensitivity percentile", config.sensitivity_target_percentile, "P90 sensitivity target retained by the pipeline."),
    ("Quantile method", config.target_quantile_method, "Loaded from the frozen 03A configuration."),
], columns=["policy", "value", "scientific_justification"])
display(evidence_table)

stage0_checks = pd.DataFrame([
    ("all_required_inputs_exist", all(path.exists() for path in required_inputs), str(required_inputs)),
    ("preprocessing_validation_passed", all(preprocessing_manifest["validation"].values()), str(preprocessing_manifest["validation"])),
    ("eda_validation_passed", all(eda_manifest["validation"].values()), str(eda_manifest["validation"])),
    ("input_signature_is_csv", input_signature_before["format"] == "csv", str(input_signature_before)),
    ("input_header_matches_contract", pd.read_csv(INTERVAL_PATH, nrows=0).columns.tolist() == INTERVAL_COLUMNS, f"{len(INTERVAL_COLUMNS)} columns"),
    ("manifest_timezone_matches", preprocessing_manifest["policy"]["timezone"] == config.timezone, config.timezone),
    ("expected_intervals_match", preprocessing_manifest["policy"]["expected_intervals_per_day"] == config.expected_intervals_per_day, str(config.expected_intervals_per_day)),
    ("frozen_target_configuration_loaded", selected_target_config.get("status") == "frozen_primary_research_policy", selected_target_config.get("configuration_id", "")),
    ("frozen_target_policy_matches_design_grid", len(target_design_candidate) == 1, f"target={config.target_day_min_intervals}; history={config.history_day_min_intervals}; window={config.target_history_window_days}; support={config.minimum_history_fraction:.0%}"),
    ("frozen_target_policy_matches_masking_evidence", len(masking_candidate) == 1 and np.isclose(float(masking_candidate.iloc[0]["label_agreement_rate"]), float(evidence["target_day_94_label_agreement"]), atol=1e-12), f"label agreement={float(masking_candidate.iloc[0]['label_agreement_rate']):.4%}"),
    ("frozen_history_evidence_matches_saved_results", len(history_policy_candidate) == 1 and int(history_policy_candidate.iloc[0]["eligible_household_days"]) == int(evidence["history_92_75_eligible_household_days"]), f"eligible days={int(history_policy_candidate.iloc[0]['eligible_household_days']):,}"),
    ("target_and_history_coverage_policies_valid", config.expected_intervals_per_day >= config.target_day_min_intervals >= config.history_day_min_intervals > 0, f"target={config.target_day_min_intervals}; history={config.history_day_min_intervals}"),
    ("history_requirement_valid", config.target_min_history_days == int(np.ceil(config.target_history_window_days * config.minimum_history_fraction)), f"{config.target_min_history_days} days = ceil({config.minimum_history_fraction:.0%} × {config.target_history_window_days})"),
    ("percentiles_valid", 0 < config.sensitivity_target_percentile < config.primary_target_percentile < 1, f"P{config.sensitivity_target_percentile:.2f}, P{config.primary_target_percentile:.2f}"),
    ("rolling_window_excludes_target_date", config.rolling_window_closed == "left", config.rolling_window_closed),
    ("interval_row_count_matches", input_signature_before["rows"] == preprocessing_manifest["primary_rows"], f"{input_signature_before['rows']:,}"),
    ("interval_hours_derived", np.isclose(INTERVAL_HOURS, .25), str(INTERVAL_HOURS)),
], columns=["check", "passed", "evidence"])
publish_table(stage0_checks, "03_00_stage_0_validation.csv", "Stage 0 validation")
failed_stage0 = stage0_checks.loc[~stage0_checks["passed"], "check"].tolist()
if failed_stage0:
    raise RuntimeError(f"Stage 0 validation failed: {failed_stage0}")

configuration_payload = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "research_stage": "household_day_and_target",
    "target_configuration_id": config.target_configuration_id,
    "target_policy_status": selected_target_config["status"],
    "target_policy_source": str(TARGET_CONFIG_PATH),
    "target_policy_decision_evidence": evidence,
    "configuration": asdict(config), "csv_policy": {"encoding": CSV_ENCODING, "separator": CSV_SEPARATOR, "missing_value": CSV_MISSING_VALUE, "chunk_size": CSV_CHUNK_SIZE},
    "derived": {"interval_hours": INTERVAL_HOURS},
    "inputs": {"interval": str(INTERVAL_PATH), "interval_dtype_contract": str(INTERVAL_DTYPE_CONTRACT_PATH), "coverage": str(COVERAGE_PATH), "preprocessing_manifest": str(PREPROCESSING_MANIFEST_PATH), "eda_manifest": str(EDA_MANIFEST_PATH), "frozen_target_config": str(TARGET_CONFIG_PATH), "target_design_results": str(TARGET_DESIGN_RESULTS_PATH), "history_policy_results": str(HISTORY_POLICY_RESULTS_PATH)},
    "outputs": {"observed_daily": str(OBSERVED_DAILY_OUTPUT_PATH), "observed_daily_dtype_contract": str(OBSERVED_DAILY_DTYPE_CONTRACT_PATH), "daily": str(DAILY_OUTPUT_PATH), "daily_dtype_contract": str(DAILY_DTYPE_CONTRACT_PATH), "target": str(TARGET_OUTPUT_PATH), "target_dtype_contract": str(TARGET_DTYPE_CONTRACT_PATH)},
    "software": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__},
}
write_json_atomic(configuration_payload, CONFIG_OUTPUT_PATH)
print(f"Saved configuration: {CONFIG_OUTPUT_PATH}")
STAGE_0_COMPLETE = True



## Stage 1 — Validate and use the existing interval dataset

In [ ]:
interval_columns = pd.read_csv(INTERVAL_PATH, nrows=0).columns.tolist()

required_columns = [
    "household_id", "measurement_timestamp_local", "measurement_date", "time_slot_index",
    "avg_import_kw", "is_valid_for_daily_aggregation", "is_missing_import_demand",
    "is_negative_import_demand", "is_zero_import", "is_off_15min_grid",
    "has_invalid_numeric_text", "duplicate_status", "is_duplicate_conflict",
    "branch", "depot", "substation_id", "network_metadata_match_type",
    "quality_status", "interval_exclusion_reason", "obis_normalized",
    "is_global_iqr_extreme_eda_only", "is_global_top_1pct_eda_only",
    "is_global_top_0_1pct_eda_only", "is_imputed",
]
missing_required_columns = sorted(set(required_columns) - set(interval_columns))
schema_table = pd.DataFrame({
    "column": interval_columns,
    "logical_dtype": [interval_dtype_contract["columns"][name] for name in interval_columns],
    "required_for_stage_1_or_2": [name in required_columns for name in interval_columns],
})
publish_table(schema_table, "03_02_required_column_validation.csv", "Interval CSV schema contract", display_rows=60)
if missing_required_columns:
    raise ValueError(f"Missing required interval columns: {missing_required_columns}")


In [ ]:
audit_columns = [
    "household_id", "measurement_date", "time_slot_index", "avg_import_kw",
    "is_valid_for_daily_aggregation", "is_missing_import_demand", "is_negative_import_demand",
    "is_off_15min_grid", "has_invalid_numeric_text", "is_duplicate_conflict",
    "obis_normalized", "is_global_iqr_extreme_eda_only", "is_global_top_1pct_eda_only",
    "is_global_top_0_1pct_eda_only", "is_imputed", "interval_exclusion_reason", "quality_status",
]
audit_counts = Counter()
exclusion_counts = Counter()
quality_counts = Counter()
obis_counts = Counter()
seen_households = set()
repeated_complete_households = set()
duplicate_interval_keys = 0
maximum_valid_import_kw = float("-inf")

for frame in iter_complete_household_chunks(audit_columns):
    audit_counts["rows"] += len(frame)
    for column in [
        "is_valid_for_daily_aggregation", "is_missing_import_demand", "is_negative_import_demand",
        "is_off_15min_grid", "has_invalid_numeric_text", "is_duplicate_conflict",
        "is_global_iqr_extreme_eda_only", "is_global_top_1pct_eda_only",
        "is_global_top_0_1pct_eda_only", "is_imputed",
    ]:
        audit_counts[column] += int(frame[column].fillna(False).astype(bool).sum())

    exclusion_counts.update(frame["interval_exclusion_reason"].fillna("<missing>").astype(str))
    quality_counts.update(frame["quality_status"].fillna("<missing>").astype(str))
    obis_counts.update(frame["obis_normalized"].fillna("<missing>").astype(str))

    valid_mask = frame["is_valid_for_daily_aggregation"].fillna(False).astype(bool)
    valid = frame.loc[valid_mask]
    if not valid.empty:
        maximum_valid_import_kw = max(maximum_valid_import_kw, float(valid["avg_import_kw"].max()))
        duplicate_interval_keys += int(valid.duplicated(["household_id", "measurement_date", "time_slot_index"], keep=False).sum())

    current_households = set(frame["household_id"].dropna().astype(str).unique())
    repeated_complete_households.update(seen_households.intersection(current_households))
    seen_households.update(current_households)

interval_input_validation = pd.DataFrame([
    ("lp_rows", audit_counts["rows"], preprocessing_manifest["counts"]["lp_rows"]),
    ("valid_aggregation_rows", audit_counts["is_valid_for_daily_aggregation"], preprocessing_manifest["counts"]["valid_aggregation_rows"]),
    ("missing_import_rows", audit_counts["is_missing_import_demand"], preprocessing_manifest["counts"]["missing_import_rows"]),
    ("negative_import_rows", audit_counts["is_negative_import_demand"], 0),
    ("off_grid_rows", audit_counts["is_off_15min_grid"], 0),
    ("invalid_numeric_text_rows", audit_counts["has_invalid_numeric_text"], 0),
    ("duplicate_conflict_rows", audit_counts["is_duplicate_conflict"], 0),
    ("imputed_rows", audit_counts["is_imputed"], 0),
    ("global_iqr_rows_retained", audit_counts["is_global_iqr_extreme_eda_only"], preprocessing_manifest["counts"]["global_iqr_extreme_rows_retained"]),
    ("global_top_1pct_rows_retained", audit_counts["is_global_top_1pct_eda_only"], preprocessing_manifest["counts"]["global_top_1pct_rows_retained"]),
    ("global_top_0_1pct_rows_retained", audit_counts["is_global_top_0_1pct_eda_only"], preprocessing_manifest["counts"]["global_top_0_1pct_rows_retained"]),
], columns=["measure", "observed", "expected"])
interval_input_validation["matches"] = interval_input_validation["observed"].eq(interval_input_validation["expected"])
publish_table(interval_input_validation, "03_01_interval_input_validation.csv", "Interval input reconciliation")

aggregation_eligibility_summary = pd.DataFrame([
    ("valid_for_daily_aggregation", audit_counts["is_valid_for_daily_aggregation"]),
    ("excluded_missing_import_demand", audit_counts["is_missing_import_demand"]),
], columns=["aggregation_status", "rows"])
aggregation_eligibility_summary["percentage"] = aggregation_eligibility_summary["rows"] / audit_counts["rows"] * 100
publish_table(aggregation_eligibility_summary, "03_03_aggregation_eligibility_summary.csv", "Aggregation eligibility")

stage1_checks = pd.DataFrame([
    ("all_csv_rows_scanned", audit_counts["rows"] == input_signature_before["rows"], f"{audit_counts['rows']:,}"),
    ("all_reconciliation_counts_match", bool(interval_input_validation["matches"].all()), str(interval_input_validation.loc[~interval_input_validation["matches"], "measure"].tolist())),
    ("only_lp_records", set(obis_counts) == {"LP"}, str(obis_counts)),
    ("no_unresolved_duplicate_conflicts", audit_counts["is_duplicate_conflict"] == 0, str(audit_counts["is_duplicate_conflict"])),
    ("no_duplicate_valid_interval_keys", duplicate_interval_keys == 0, str(duplicate_interval_keys)),
    ("households_emitted_once_after_chunk_carry", len(repeated_complete_households) == 0, f"repeated households={len(repeated_complete_households)}"),
    ("valid_rows_have_nonmissing_demand", audit_counts["is_valid_for_daily_aggregation"] + audit_counts["is_missing_import_demand"] == audit_counts["rows"], str(exclusion_counts)),
    ("maximum_high_value_retained", np.isclose(maximum_valid_import_kw, 43.2648, atol=1e-9), str(maximum_valid_import_kw)),
    ("no_interval_imputation", audit_counts["is_imputed"] == 0, str(audit_counts["is_imputed"])),
], columns=["check", "passed", "evidence"])
publish_table(stage1_checks, "03_01_stage_1_validation.csv", "Stage 1 validation")
failed_stage1 = stage1_checks.loc[~stage1_checks["passed"], "check"].tolist()
if failed_stage1:
    raise RuntimeError(f"Stage 1 validation failed: {failed_stage1}")

del frame, valid
STAGE_1_COMPLETE = True


## Stage 2 — Create the household-day dataset

In [ ]:
aggregation_columns = [
    "household_id", "measurement_date", "measurement_timestamp_local", "avg_import_kw", "is_valid_for_daily_aggregation",
    "is_zero_import", "branch", "depot", "substation_id", "network_metadata_match_type",
]
daily_demand_frames = []
daily_network_frames = []

for frame in iter_complete_household_chunks(aggregation_columns):
    frame["household_id"] = frame["household_id"].astype("string")
    frame["measurement_date"] = pd.to_datetime(frame["measurement_date"], errors="raise").dt.normalize()
    keys = ["household_id", "measurement_date"]

    valid_mask = frame["is_valid_for_daily_aggregation"].fillna(False).astype(bool)
    valid = frame.loc[valid_mask].copy()
    if not valid.empty:
        daily_demand = valid.groupby(keys, sort=False, observed=True).agg(
            observed_daily_peak_kw=("avg_import_kw", "max"),
            observed_daily_mean_kw=("avg_import_kw", "mean"),
            observed_daily_median_kw=("avg_import_kw", "median"),
            observed_daily_std_kw=("avg_import_kw", lambda values: values.std(ddof=0)),
            observed_interval_p90_kw=("avg_import_kw", lambda values: values.quantile(0.90, interpolation="linear")),
            observed_interval_p95_kw=("avg_import_kw", lambda values: values.quantile(0.95, interpolation="linear")),
            observed_daily_min_kw=("avg_import_kw", "min"),
            observed_import_power_sum_kw=("avg_import_kw", "sum"),
            valid_interval_count_aggregated=("avg_import_kw", "size"),
            zero_interval_count=("is_zero_import", "sum"),
        ).reset_index()
        daily_demand["observed_import_energy_kwh"] = daily_demand["observed_import_power_sum_kw"] * INTERVAL_HOURS
        peak_rows = valid.loc[valid.groupby(keys, sort=False, observed=True)["avg_import_kw"].idxmax(), [*keys, "measurement_timestamp_local"]].copy()
        peak_rows["observed_peak_hour"] = pd.to_datetime(peak_rows["measurement_timestamp_local"], errors="raise").dt.hour.astype("int8")
        daily_demand = daily_demand.merge(peak_rows[[*keys, "observed_peak_hour"]], on=keys, how="left", validate="one_to_one")
        daily_demand["observed_peak_hour_sin"] = np.sin(2 * np.pi * daily_demand["observed_peak_hour"] / 24.0)
        daily_demand["observed_peak_hour_cos"] = np.cos(2 * np.pi * daily_demand["observed_peak_hour"] / 24.0)
        daily_demand["observed_load_factor"] = daily_demand["observed_daily_mean_kw"] / daily_demand["observed_daily_peak_kw"].where(daily_demand["observed_daily_peak_kw"].gt(0))
        daily_demand["observed_import_active_count"] = daily_demand["valid_interval_count_aggregated"] - daily_demand["zero_interval_count"]
        daily_demand["observed_import_active_ratio"] = daily_demand["observed_import_active_count"] / daily_demand["valid_interval_count_aggregated"]
        daily_demand["zero_interval_ratio"] = daily_demand["zero_interval_count"] / daily_demand["valid_interval_count_aggregated"]
        daily_demand_frames.append(daily_demand)

    frame["metadata_is_backward_valid"] = frame["network_metadata_match_type"].astype(str).eq("backward_valid")
    network_base = frame.groupby(keys, sort=False, observed=True).agg(
        network_observed_rows=("household_id", "size"),
        network_backward_valid_rows=("metadata_is_backward_valid", "sum"),
    ).reset_index()

    valid_metadata = frame.loc[frame["metadata_is_backward_valid"]].copy()
    if not valid_metadata.empty:
        network_values = valid_metadata.groupby(keys, sort=False, observed=True).agg(
            candidate_branch=("branch", "first"), candidate_depot=("depot", "first"),
            candidate_substation_id=("substation_id", "first"), branch_distinct_count=("branch", "nunique"),
            depot_distinct_count=("depot", "nunique"), substation_distinct_count=("substation_id", "nunique"),
        ).reset_index()
        network_base = network_base.merge(network_values, on=keys, how="left", validate="one_to_one")
    else:
        for column in ["candidate_branch", "candidate_depot", "candidate_substation_id", "branch_distinct_count", "depot_distinct_count", "substation_distinct_count"]:
            network_base[column] = pd.NA
    daily_network_frames.append(network_base)

observed_demand = pd.concat(daily_demand_frames, ignore_index=True)
observed_network = pd.concat(daily_network_frames, ignore_index=True)

if observed_demand.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Daily demand keys crossed household-safe CSV chunks.")
if observed_network.duplicated(["household_id", "measurement_date"]).any():
    raise RuntimeError("Daily network keys crossed household-safe CSV chunks.")

print(f"Observed daily demand groups: {len(observed_demand):,}")
print(f"Observed daily network groups: {len(observed_network):,}")


In [ ]:
coverage_dtypes = {
    "household_id": "string", "raw_lp_rows": "Int64", "unique_slot_count": "Int64",
    "valid_interval_count": "Int64", "missing_import_count": "Int64", "duplicate_conflict_count": "Int64",
    "statistical_extreme_count": "Int64", "coverage_ratio": "float64", "coverage_status": "string",
    "is_valid_primary_target_day": "boolean",
}
coverage = pd.read_csv(
    COVERAGE_PATH, dtype=coverage_dtypes, parse_dates=["measurement_date"],
    true_values=["True"], false_values=["False"], keep_default_na=False, na_values=[""],
    encoding=CSV_ENCODING, low_memory=False,
)
coverage["household_id"] = coverage["household_id"].astype("string")
coverage["measurement_date"] = pd.to_datetime(coverage["measurement_date"], errors="raise").dt.normalize()
keys = ["household_id", "measurement_date"]

if coverage.duplicated(keys).any():
    raise RuntimeError("Coverage audit contains duplicate household-date keys.")

observed_daily = coverage.merge(observed_demand, on=keys, how="left", validate="one_to_one")
observed_daily = observed_daily.merge(observed_network, on=keys, how="left", validate="one_to_one")

observed_daily["valid_interval_count_aggregated"] = observed_daily["valid_interval_count_aggregated"].fillna(0).astype("int32")
interval_count_mismatches = int((observed_daily["valid_interval_count"] != observed_daily["valid_interval_count_aggregated"]).sum())
if interval_count_mismatches:
    raise RuntimeError(f"Coverage and aggregation interval counts differ for {interval_count_mismatches:,} household-days.")

observed_daily["network_uncertain_rows"] = (
    observed_daily["network_observed_rows"] - observed_daily["network_backward_valid_rows"]
).fillna(observed_daily["network_observed_rows"]).astype("int32")
for column in ["branch_distinct_count", "depot_distinct_count", "substation_distinct_count"]:
    observed_daily[column] = observed_daily[column].fillna(0).astype("int16")

stable_network = (
    observed_daily["network_uncertain_rows"].eq(0)
    & observed_daily["branch_distinct_count"].eq(1)
    & observed_daily["depot_distinct_count"].eq(1)
    & observed_daily["substation_distinct_count"].eq(1)
)
observed_daily["daily_network_metadata_status"] = np.where(stable_network, "valid_stable", "unknown_or_unstable")
observed_daily["branch"] = observed_daily["candidate_branch"].where(stable_network, "Unknown").astype("string")
observed_daily["depot"] = observed_daily["candidate_depot"].where(stable_network, "Unknown").astype("string")
observed_daily["substation_id"] = observed_daily["candidate_substation_id"].where(stable_network, "Unknown").astype("string")
observed_daily["network_metadata_valid_ratio"] = (
    observed_daily["network_backward_valid_rows"] / observed_daily["network_observed_rows"]
).fillna(0.0)
observed_daily["day_observed"] = True
write_csv_atomic(observed_daily, OBSERVED_DAILY_OUTPUT_PATH)
observed_daily_dtype_contract = write_dtype_contract(observed_daily, OBSERVED_DAILY_OUTPUT_PATH, OBSERVED_DAILY_DTYPE_CONTRACT_PATH)
observed_daily_signature = csv_signature(OBSERVED_DAILY_OUTPUT_PATH, len(observed_daily), len(observed_daily.columns))
observed_daily_roundtrip = read_dataset_csv(OBSERVED_DAILY_OUTPUT_PATH, observed_daily_dtype_contract, usecols=["household_id", "measurement_date", "valid_interval_count"])
observed_daily_roundtrip_passed = len(observed_daily_roundtrip) == len(observed_daily) == len(coverage) and not observed_daily_roundtrip.duplicated(["household_id", "measurement_date"]).any()

calendar_parts = []
for household_id, group in observed_daily.groupby("household_id", sort=False, observed=True):
    dates = pd.date_range(group["measurement_date"].min(), group["measurement_date"].max(), freq="D")
    calendar_parts.append(pd.DataFrame({"household_id": household_id, "measurement_date": dates}))
household_calendar = pd.concat(calendar_parts, ignore_index=True)

daily = household_calendar.merge(observed_daily, on=keys, how="left", validate="one_to_one")
daily["day_observed"] = daily["day_observed"].fillna(False).astype(bool)
daily["is_full_missing_day"] = ~daily["day_observed"]

count_columns = [
    "raw_lp_rows", "unique_slot_count", "valid_interval_count", "duplicate_conflict_count",
    "statistical_extreme_count", "valid_interval_count_aggregated", "zero_interval_count",
    "network_observed_rows", "network_backward_valid_rows", "network_uncertain_rows",
    "branch_distinct_count", "depot_distinct_count", "substation_distinct_count",
]
for column in count_columns:
    daily[column] = daily[column].fillna(0).astype("int32")
daily["missing_import_count"] = daily["missing_import_count"].where(daily["day_observed"], config.expected_intervals_per_day).fillna(0).astype("int32")
daily["coverage_ratio"] = daily["coverage_ratio"].fillna(0.0).astype("float64")
daily["network_metadata_valid_ratio"] = daily["network_metadata_valid_ratio"].fillna(0.0).astype("float64")
daily["daily_network_metadata_status"] = daily["daily_network_metadata_status"].fillna("full_day_missing").astype("string")
for column in ["branch", "depot", "substation_id"]:
    daily[column] = daily[column].fillna("Unknown").astype("string")

daily["expected_missing_interval_count"] = config.expected_intervals_per_day - daily["valid_interval_count"]
daily["is_complete_day"] = daily["day_observed"] & daily["valid_interval_count"].eq(config.expected_intervals_per_day)
daily["is_near_complete_day"] = daily["day_observed"] & daily["valid_interval_count"].between(config.near_complete_min_intervals, config.expected_intervals_per_day - 1)
daily["is_severely_incomplete_day"] = daily["day_observed"] & daily["valid_interval_count"].lt(config.near_complete_min_intervals)
daily["completeness_category"] = np.select(
    [daily["is_full_missing_day"], daily["is_complete_day"], daily["is_near_complete_day"]],
    ["full_day_missing", "complete_96", "near_complete_70_95"],
    default="severely_incomplete_below_70",
)
daily["is_daily_peak_complete"] = daily["is_complete_day"]
daily["is_daily_energy_complete"] = daily["is_complete_day"]
daily["is_valid_history_day"] = daily["day_observed"] & daily["valid_interval_count"].ge(config.history_day_min_intervals)
daily["is_valid_target_day"] = daily["day_observed"] & daily["valid_interval_count"].ge(config.target_day_min_intervals)

observed_metric_columns = {
    "observed_daily_peak_kw": "daily_peak_kw",
    "observed_daily_mean_kw": "daily_mean_kw",
    "observed_daily_median_kw": "daily_median_kw",
    "observed_daily_std_kw": "daily_std_kw",
    "observed_daily_min_kw": "daily_min_kw",
    "observed_import_energy_kwh": "daily_import_energy_kwh",
}
for observed_column, complete_column in observed_metric_columns.items():
    daily[complete_column] = daily[observed_column].where(daily["is_complete_day"])
daily["daily_peak_kw"] = daily["observed_daily_peak_kw"].where(daily["is_valid_history_day"])
daily["target_day_peak_kw"] = daily["observed_daily_peak_kw"].where(daily["is_valid_target_day"])

daily = daily.sort_values(keys, kind="mergesort").reset_index(drop=True)
print(f"Observed household-days: {daily['day_observed'].sum():,}")
print(f"Continuous household-day rows: {len(daily):,}")
print(f"Full missing days inserted: {daily['is_full_missing_day'].sum():,}")

In [ ]:
continuity_gap_count = 0
for _, group in daily.groupby("household_id", sort=False, observed=True):
    differences = group["measurement_date"].diff().dropna().dt.days
    continuity_gap_count += int(differences.ne(1).sum())

energy_identity = (
    daily["observed_daily_mean_kw"]
    * daily["valid_interval_count"]
    * INTERVAL_HOURS
)
energy_rows = daily["observed_import_energy_kwh"].notna()
energy_matches = np.allclose(
    daily.loc[energy_rows, "observed_import_energy_kwh"],
    energy_identity.loc[energy_rows],
    rtol=1e-10,
    atol=1e-10,
)

input_signature_stage2 = input_signature(INTERVAL_PATH, preprocessing_manifest)
stage2_checks = pd.DataFrame([
    ("household_date_keys_unique", not daily.duplicated(keys).any(), str(int(daily.duplicated(keys).sum()))),
    ("calendar_continuity", continuity_gap_count == 0, str(continuity_gap_count)),
    ("observed_days_reconcile", int(daily["day_observed"].sum()) == len(coverage) == eda_manifest["population"]["household_days"], f"{int(daily['day_observed'].sum()):,}"),
    ("valid_interval_counts_reconcile", int(daily["valid_interval_count"].sum()) == preprocessing_manifest["counts"]["valid_aggregation_rows"], f"{int(daily['valid_interval_count'].sum()):,}"),
    ("coverage_aggregation_counts_match", interval_count_mismatches == 0, str(interval_count_mismatches)),
    ("observed_daily_csv_roundtrip", observed_daily_roundtrip_passed, f"{len(observed_daily_roundtrip):,} observed household-days"),
    ("complete_days_have_96_intervals", daily.loc[daily["is_complete_day"], "valid_interval_count"].eq(96).all(), str(int(daily['is_complete_day'].sum()))),
    ("complete_days_reconcile", int(daily["is_complete_day"].sum()) == int(coverage["valid_interval_count"].eq(config.expected_intervals_per_day).sum()), f"{int(daily['is_complete_day'].sum()):,}"),
    ("full_missing_days_have_no_demand", daily.loc[daily["is_full_missing_day"], list(observed_metric_columns)].isna().all().all(), str(int(daily['is_full_missing_day'].sum()))),
    ("daily_energy_identity", bool(energy_matches), f"interval_hours={INTERVAL_HOURS}"),
    ("maximum_high_value_retained", np.isclose(daily["observed_daily_peak_kw"].max(), maximum_valid_import_kw, atol=1e-9), str(daily['observed_daily_peak_kw'].max())),
    ("network_values_only_when_stable", daily.loc[daily["daily_network_metadata_status"].ne("valid_stable"), "branch"].eq("Unknown").all(), str(daily['daily_network_metadata_status'].value_counts().to_dict())),
    ("interval_input_unchanged", input_signature_stage2 == input_signature_before, str(input_signature_stage2)),
], columns=["check", "passed", "evidence"])
publish_table(stage2_checks, "03_02_stage_2_validation.csv", "Stage 2 validation")
failed_stage2 = stage2_checks.loc[~stage2_checks["passed"], "check"].tolist()
if failed_stage2:
    raise RuntimeError(f"Stage 2 validation failed: {failed_stage2}")

population_summary = pd.DataFrame([
    ("households", daily["household_id"].nunique()),
    ("observed_household_days", int(daily["day_observed"].sum())),
    ("continuous_calendar_rows", len(daily)),
    ("full_missing_days_inserted", int(daily["is_full_missing_day"].sum())),
    ("target_day_candidates_at_configured_coverage", int(daily["is_valid_target_day"].sum())),
    ("history_day_candidates_at_configured_coverage", int(daily["is_valid_history_day"].sum())),
], columns=["measure", "value"])
completeness_summary = daily["completeness_category"].value_counts().rename_axis("completeness_category").reset_index(name="household_days")
completeness_summary["percentage"] = completeness_summary["household_days"] / len(daily) * 100
exact_coverage_summary = daily.loc[daily["day_observed"]].groupby("valid_interval_count", as_index=False, observed=True).size().rename(columns={"size": "observed_household_days"})
exact_coverage_summary["target_candidate_at_94_plus"] = exact_coverage_summary["valid_interval_count"].ge(config.target_day_min_intervals)
exact_coverage_summary["history_candidate_at_92_plus"] = exact_coverage_summary["valid_interval_count"].ge(config.history_day_min_intervals)
full_missing_summary = daily.groupby("household_id", observed=True)["is_full_missing_day"].sum().describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).rename("full_missing_days").reset_index()
daily_metric_summary = daily[[*observed_metric_columns.keys(), *observed_metric_columns.values()]].describe(percentiles=[0.01, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).T.reset_index(names="metric")
network_summary = daily["daily_network_metadata_status"].value_counts().rename_axis("daily_network_metadata_status").reset_index(name="household_days")

publish_table(population_summary, "03_04_household_day_population_summary.csv", "Household-day population")
publish_table(completeness_summary, "03_05_daily_completeness_summary.csv", "Daily completeness")
publish_table(exact_coverage_summary, "03_05a_exact_interval_coverage_distribution.csv", "Exact observed interval-count distribution", display_rows=30)
publish_table(full_missing_summary, "03_06_full_missing_day_summary.csv", "Full missing-day distribution")
publish_table(daily_metric_summary, "03_07_daily_metric_summary.csv", "Daily metric summary", display_rows=20)
publish_table(network_summary, "03_08_daily_network_metadata_summary.csv", "Daily network metadata")

daily_output_columns = [
    "household_id", "measurement_date", "day_observed", "is_full_missing_day",
    "raw_lp_rows", "unique_slot_count", "valid_interval_count", "missing_import_count",
    "expected_missing_interval_count", "duplicate_conflict_count", "statistical_extreme_count",
    "coverage_ratio", "completeness_category", "is_complete_day", "is_near_complete_day",
    "is_severely_incomplete_day", "is_daily_peak_complete", "is_daily_energy_complete",
    "is_valid_history_day", "is_valid_target_day", "observed_daily_peak_kw", "target_day_peak_kw", "observed_daily_mean_kw",
    "observed_daily_median_kw", "observed_daily_std_kw", "observed_daily_min_kw",
    "observed_interval_p90_kw", "observed_interval_p95_kw", "observed_peak_hour",
    "observed_peak_hour_sin", "observed_peak_hour_cos", "observed_load_factor",
    "observed_import_active_count", "observed_import_active_ratio",
    "observed_import_energy_kwh", "zero_interval_count", "zero_interval_ratio",
    "daily_peak_kw", "daily_mean_kw", "daily_median_kw", "daily_std_kw", "daily_min_kw",
    "daily_import_energy_kwh", "branch", "depot", "substation_id",
    "daily_network_metadata_status", "network_metadata_valid_ratio", "network_observed_rows",
    "network_backward_valid_rows", "network_uncertain_rows", "branch_distinct_count",
    "depot_distinct_count", "substation_distinct_count",
]
daily = daily[daily_output_columns]
write_csv_atomic(daily, DAILY_OUTPUT_PATH)
daily_dtype_contract = write_dtype_contract(daily, DAILY_OUTPUT_PATH, DAILY_DTYPE_CONTRACT_PATH)
daily_signature = csv_signature(DAILY_OUTPUT_PATH, len(daily), len(daily.columns))

daily_missingness_after = pd.Series(0, index=daily.columns, dtype="int64")
daily_roundtrip_rows = 0
for roundtrip_chunk in read_dataset_csv(DAILY_OUTPUT_PATH, daily_dtype_contract, chunksize=100_000):
    daily_roundtrip_rows += len(roundtrip_chunk)
    daily_missingness_after = daily_missingness_after.add(roundtrip_chunk.isna().sum(), fill_value=0).astype("int64")

daily_roundtrip_columns = [
    "household_id", "measurement_date", "valid_interval_count", "observed_daily_mean_kw",
    "observed_import_energy_kwh", "daily_import_energy_kwh", "is_complete_day",
]
daily_roundtrip = read_dataset_csv(DAILY_OUTPUT_PATH, daily_dtype_contract, usecols=daily_roundtrip_columns)
roundtrip_energy = daily_roundtrip["observed_daily_mean_kw"] * daily_roundtrip["valid_interval_count"] * INTERVAL_HOURS
roundtrip_energy_rows = daily_roundtrip["observed_import_energy_kwh"].notna()
daily_csv_checks = pd.DataFrame([
    ("daily_csv_row_count", daily_roundtrip_rows == len(daily), f"{daily_roundtrip_rows:,}"),
    ("daily_csv_households", daily_roundtrip["household_id"].nunique() == daily["household_id"].nunique(), str(daily_roundtrip["household_id"].nunique())),
    ("daily_csv_keys_unique", not daily_roundtrip.duplicated(["household_id", "measurement_date"]).any(), str(int(daily_roundtrip.duplicated(["household_id", "measurement_date"]).sum()))),
    ("daily_csv_missingness_matches", daily_missingness_after.equals(daily.isna().sum().astype("int64")), str((daily_missingness_after - daily.isna().sum()).abs().sum())),
    ("daily_energy_identity_after_csv", np.allclose(daily_roundtrip.loc[roundtrip_energy_rows, "observed_import_energy_kwh"], roundtrip_energy.loc[roundtrip_energy_rows], rtol=1e-12, atol=1e-12), f"interval_hours={INTERVAL_HOURS}"),
    ("daily_complete_energy_roundtrip", np.allclose(daily_roundtrip["daily_import_energy_kwh"], daily["daily_import_energy_kwh"], equal_nan=True, rtol=1e-12, atol=1e-12), "float round-trip"),
    ("daily_dtype_contract_written", DAILY_DTYPE_CONTRACT_PATH.exists(), str(DAILY_DTYPE_CONTRACT_PATH)),
], columns=["check", "passed", "evidence"])
publish_table(daily_csv_checks, "03_02_csv_roundtrip_validation.csv", "Daily CSV round-trip validation")
if not daily_csv_checks["passed"].all():
    raise RuntimeError(f"Daily CSV round-trip validation failed: {daily_csv_checks.loc[~daily_csv_checks.passed, 'check'].tolist()}")
print(f"Saved household-day dataset: {DAILY_OUTPUT_PATH} ({len(daily):,} rows)")
STAGE_2_COMPLETE = True

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
sns.barplot(data=completeness_summary, x="completeness_category", y="household_days", color="#4C78A8", ax=axes[0, 0])
axes[0, 0].set_title("Continuous-calendar household-day completeness")
axes[0, 0].tick_params(axis="x", rotation=25)
axes[0, 0].set_xlabel("")

complete_peak = daily.loc[daily["is_complete_day"], "daily_peak_kw"]
axes[0, 1].hist(np.log1p(complete_peak), bins=80, color="#F58518")
axes[0, 1].set_title("Complete-day peak distribution")
axes[0, 1].set_xlabel("log(1 + daily peak kW)")

complete_energy = daily.loc[daily["is_complete_day"], "daily_import_energy_kwh"]
axes[1, 0].hist(np.log1p(complete_energy), bins=80, color="#54A24B")
axes[1, 0].set_title("Complete-day imported energy distribution")
axes[1, 0].set_xlabel("log(1 + daily imported energy kWh)")

missing_per_household = daily.groupby("household_id", observed=True)["is_full_missing_day"].sum()
axes[1, 1].hist(missing_per_household, bins=50, color="#E45756")
axes[1, 1].set_title("Inserted full missing days per household")
axes[1, 1].set_xlabel("Full missing calendar days")
fig.suptitle("Household-day construction audit", fontsize=15)
fig.tight_layout()
publish_figure(fig, "03_01_household_day_construction_dashboard.png")

## Stage 3 — Construct the leakage-free household-relative target
The frozen 03A configuration supplies the target-day and history-day coverage rules, calendar window, minimum history support, primary quantile, and quantile method. Only qualifying prior days contribute to the rolling thresholds. The target date is excluded using the configured left-closed rolling window. P95 is primary and P90 is retained as a sensitivity label.

In [ ]:
daily_target = daily.sort_values(["household_id", "measurement_date"], kind="mergesort").reset_index(drop=True).copy()
target_parts = []
window = f"{config.target_history_window_days}D"

for household_id, group in daily_target.groupby("household_id", sort=False, observed=True):
    group = group.sort_values("measurement_date", kind="mergesort").copy()
    historical_peaks = group.set_index("measurement_date")["daily_peak_kw"]
    rolling_history = historical_peaks.rolling(window, closed=config.rolling_window_closed)
    group["historical_valid_day_count_45d"] = rolling_history.count().fillna(0).to_numpy(dtype="int16")
    group["historical_peak_p95_45d"] = rolling_history.quantile(
        config.primary_target_percentile,
        interpolation=config.target_quantile_method,
    ).to_numpy()
    group["historical_peak_p90_45d"] = rolling_history.quantile(
        config.sensitivity_target_percentile,
        interpolation=config.target_quantile_method,
    ).to_numpy()
    target_parts.append(group)

daily_target = pd.concat(target_parts, ignore_index=True)
daily_target["target_config_id"] = config.target_configuration_id
daily_target["historical_window_start"] = daily_target["measurement_date"] - pd.Timedelta(days=config.target_history_window_days)
daily_target["historical_window_end"] = daily_target["measurement_date"] - pd.Timedelta(days=1)
daily_target["has_sufficient_target_history"] = daily_target["historical_valid_day_count_45d"].ge(config.target_min_history_days)
daily_target["has_sensitivity_history"] = daily_target["historical_valid_day_count_45d"].ge(config.sensitivity_history_days[1])
daily_target["target_eligible"] = (
    daily_target["is_valid_target_day"]
    & daily_target["has_sufficient_target_history"]
    & daily_target["target_day_peak_kw"].notna()
    & daily_target["historical_peak_p95_45d"].notna()
)

daily_target["target_ineligibility_reason"] = np.select(
    [
        daily_target["is_full_missing_day"],
        daily_target["day_observed"] & ~daily_target["is_valid_target_day"],
        daily_target["is_valid_target_day"] & daily_target["target_day_peak_kw"].isna(),
        daily_target["is_valid_target_day"] & ~daily_target["has_sufficient_target_history"],
        daily_target["is_valid_target_day"] & daily_target["has_sufficient_target_history"] & daily_target["historical_peak_p95_45d"].isna(),
    ],
    [
        "full_day_missing", "target_day_incomplete", "missing_daily_peak",
        "insufficient_valid_45d_history", "missing_historical_threshold",
    ],
    default="eligible",
)

for target_column, threshold_column in [
    ("target_high_demand_p95", "historical_peak_p95_45d"),
    ("target_high_demand_p90", "historical_peak_p90_45d"),
]:
    daily_target[target_column] = pd.Series(pd.NA, index=daily_target.index, dtype="Int8")
    daily_target.loc[daily_target["target_eligible"], target_column] = (
        daily_target.loc[daily_target["target_eligible"], "target_day_peak_kw"]
        > daily_target.loc[daily_target["target_eligible"], threshold_column]
    ).astype("int8")

daily_target["target_audit_peak_above_p95_kw"] = (
    daily_target["target_day_peak_kw"] - daily_target["historical_peak_p95_45d"]
).where(daily_target["target_eligible"])
safe_p95 = daily_target["historical_peak_p95_45d"].where(daily_target["historical_peak_p95_45d"].gt(0))
daily_target["target_audit_peak_over_p95_ratio"] = (
    daily_target["target_day_peak_kw"] / safe_p95
).where(daily_target["target_eligible"])

print(f"Primary target-eligible rows: {daily_target['target_eligible'].sum():,}")
print(f"P95 events: {int(daily_target['target_high_demand_p95'].sum(skipna=True)):,}")
print(f"P90 events: {int(daily_target['target_high_demand_p90'].sum(skipna=True)):,}")

In [ ]:
eligible_sample = daily_target.loc[daily_target["target_eligible"]].sample(
    n=min(250, int(daily_target["target_eligible"].sum())),
    random_state=config.random_state,
)
daily_groups = {household_id: group for household_id, group in daily_target.groupby("household_id", sort=False, observed=True)}
sampled_recomputation_errors = []
for row in eligible_sample.itertuples(index=False):
    household = daily_groups[row.household_id]
    historical = household.loc[
        household["measurement_date"].ge(row.measurement_date - pd.Timedelta(days=config.target_history_window_days))
        & household["measurement_date"].lt(row.measurement_date)
        & household["is_valid_history_day"],
        "daily_peak_kw",
    ].dropna()
    expected_count = len(historical)
    expected_p95 = np.quantile(historical.to_numpy(), config.primary_target_percentile, method=config.target_quantile_method)
    expected_p90 = np.quantile(historical.to_numpy(), config.sensitivity_target_percentile, method=config.target_quantile_method)
    if (
        expected_count != row.historical_valid_day_count_45d
        or not np.isclose(expected_p95, row.historical_peak_p95_45d, atol=1e-12)
        or not np.isclose(expected_p90, row.historical_peak_p90_45d, atol=1e-12)
    ):
        sampled_recomputation_errors.append((row.household_id, row.measurement_date))

eligible_minimum = daily_target["is_valid_target_day"] & daily_target["historical_valid_day_count_45d"].ge(config.target_min_history_days)
eligible_sensitivity = daily_target["is_valid_target_day"] & daily_target["historical_valid_day_count_45d"].ge(config.sensitivity_history_days[1])
ineligible_targets_missing = daily_target.loc[~daily_target["target_eligible"], ["target_high_demand_p95", "target_high_demand_p90"]].isna().all().all()
eligible_targets_valid = daily_target.loc[daily_target["target_eligible"], ["target_high_demand_p95", "target_high_demand_p90"]].isin([0, 1]).all().all()
reasons_reconcile = int(daily_target["target_ineligibility_reason"].value_counts().sum()) == len(daily_target)

target_key_columns = [
    "household_id", "measurement_date", "historical_valid_day_count_45d",
    "historical_peak_p95_45d", "historical_peak_p90_45d", "target_eligible",
    "target_high_demand_p95", "target_high_demand_p90",
]
target_key_hash_values = pd.util.hash_pandas_object(daily_target[target_key_columns], index=False).to_numpy().tobytes()
target_key_checksum = hashlib.sha256(target_key_hash_values).hexdigest()
comparable_previous_run = (
    previous_run_manifest is not None
    and previous_run_manifest.get("input_signature") == input_signature_before
    and previous_run_manifest.get("configuration") == asdict(config)
)
previous_checksum_matches = (
    not comparable_previous_run
    or previous_run_manifest.get("target_key_checksum") == target_key_checksum
)

stage3_checks = pd.DataFrame([
    ("row_count_matches_daily_dataset", len(daily_target) == len(daily), f"{len(daily_target):,}"),
    ("household_date_keys_unique", not daily_target.duplicated(["household_id", "measurement_date"]).any(), str(int(daily_target.duplicated(["household_id", "measurement_date"]).sum()))),
    ("eligibility_reasons_reconcile", reasons_reconcile, str(daily_target["target_ineligibility_reason"].value_counts().to_dict())),
    ("configured_target_policy_reproduced", np.array_equal(eligible_minimum.to_numpy(), daily_target["target_eligible"].to_numpy()), f"eligible={int(eligible_minimum.sum()):,}"),
    ("target_candidates_meet_minimum_coverage", daily_target.loc[daily_target["is_valid_target_day"], "valid_interval_count"].ge(config.target_day_min_intervals).all(), f">={config.target_day_min_intervals}/96"),
    ("history_counts_use_only_qualified_prior_days", daily_target["historical_valid_day_count_45d"].between(0, config.target_history_window_days).all(), f"history interval cutoff={config.history_day_min_intervals}; window={config.target_history_window_days}"),
    ("frozen_configuration_id_propagated", daily_target["target_config_id"].eq(config.target_configuration_id).all() if "target_config_id" in daily_target else True, config.target_configuration_id),
    ("ineligible_rows_have_missing_targets", bool(ineligible_targets_missing), "P95 and P90 are nullable"),
    ("eligible_rows_have_binary_targets", bool(eligible_targets_valid), "targets are in {0,1}"),
    ("eligible_rows_have_thresholds", daily_target.loc[daily_target["target_eligible"], ["historical_peak_p95_45d", "historical_peak_p90_45d"]].notna().all().all(), "P95 and P90 present"),
    ("target_peak_and_history_cutoffs_are_respected", daily_target.loc[daily_target["target_eligible"], "target_day_peak_kw"].notna().all() and daily_target.loc[daily_target["target_eligible"], "historical_valid_day_count_45d"].ge(config.target_min_history_days).all(), f"{config.target_day_min_intervals}/96 target; at least {config.target_min_history_days} qualifying history days"),
    ("p95_events_reconcile_to_target_values", int(daily_target["target_high_demand_p95"].sum(skipna=True)) == int((daily_target.loc[daily_target["target_eligible"], "target_day_peak_kw"] > daily_target.loc[daily_target["target_eligible"], "historical_peak_p95_45d"]).sum()), str(int(daily_target["target_high_demand_p95"].sum(skipna=True)))),
    ("p90_events_reconcile_to_target_values", int(daily_target["target_high_demand_p90"].sum(skipna=True)) == int((daily_target.loc[daily_target["target_eligible"], "target_day_peak_kw"] > daily_target.loc[daily_target["target_eligible"], "historical_peak_p90_45d"]).sum()), str(int(daily_target["target_high_demand_p90"].sum(skipna=True)))),
    ("p95_implies_p90", not (daily_target["target_high_demand_p95"].eq(1) & ~daily_target["target_high_demand_p90"].eq(1)).any(), "nested thresholds"),
    ("p90_events_not_fewer_than_p95", int(daily_target["target_high_demand_p90"].sum(skipna=True)) >= int(daily_target["target_high_demand_p95"].sum(skipna=True)), f"P90={int(daily_target['target_high_demand_p90'].sum(skipna=True)):,}; P95={int(daily_target['target_high_demand_p95'].sum(skipna=True)):,}"),
    ("sampled_past_only_recomputation_matches", len(sampled_recomputation_errors) == 0, str(sampled_recomputation_errors[:10])),
    ("reexecution_checksum_matches_when_comparable", previous_checksum_matches, "first run or matching prior checksum"),
], columns=["check", "passed", "evidence"])
publish_table(stage3_checks, "03_03_stage_3_validation.csv", "Stage 3 validation")
failed_stage3 = stage3_checks.loc[~stage3_checks["passed"], "check"].tolist()
if failed_stage3:
    raise RuntimeError(f"Stage 3 validation failed: {failed_stage3}")

In [ ]:
target_eligibility_summary = pd.DataFrame([
    ("eligible", int(daily_target["target_eligible"].sum())),
    ("ineligible", int((~daily_target["target_eligible"]).sum())),
], columns=["target_eligibility", "household_days"])
target_eligibility_summary["percentage"] = target_eligibility_summary["household_days"] / len(daily_target) * 100

ineligibility_summary = daily_target["target_ineligibility_reason"].value_counts().rename_axis("reason").reset_index(name="household_days")
ineligibility_summary["percentage"] = ineligibility_summary["household_days"] / len(daily_target) * 100

def prevalence_table(target_column, label):
    eligible = daily_target.loc[daily_target["target_eligible"], target_column].dropna().astype(int)
    return pd.DataFrame([{
        "target_definition": label,
        "eligible_rows": len(eligible),
        "positive_events": int(eligible.sum()),
        "negative_events": int((eligible == 0).sum()),
        "positive_event_rate": float(eligible.mean()),
    }])

prevalence_p95 = prevalence_table("target_high_demand_p95", "rolling_p95_previous_45_calendar_days")
prevalence_p90 = prevalence_table("target_high_demand_p90", "rolling_p90_previous_45_calendar_days")
target_by_household = daily_target.loc[daily_target["target_eligible"]].groupby("household_id", observed=True).agg(
    eligible_days=("measurement_date", "size"),
    p95_events=("target_high_demand_p95", "sum"),
    p90_events=("target_high_demand_p90", "sum"),
    median_historical_p95_kw=("historical_peak_p95_45d", "median"),
).reset_index()
target_by_household["p95_event_rate"] = target_by_household["p95_events"] / target_by_household["eligible_days"]
target_by_household["p90_event_rate"] = target_by_household["p90_events"] / target_by_household["eligible_days"]

daily_target["target_month"] = daily_target["measurement_date"].dt.to_period("M").astype(str)
target_by_month = daily_target.loc[daily_target["target_eligible"]].groupby("target_month", observed=True).agg(
    eligible_days=("measurement_date", "size"),
    p95_events=("target_high_demand_p95", "sum"),
    p90_events=("target_high_demand_p90", "sum"),
).reset_index()
target_by_month["p95_event_rate"] = target_by_month["p95_events"] / target_by_month["eligible_days"]
target_by_month["p90_event_rate"] = target_by_month["p90_events"] / target_by_month["eligible_days"]

history_sensitivity = pd.DataFrame([
    (threshold, int((daily_target["is_valid_target_day"] & daily_target["historical_valid_day_count_45d"].ge(threshold)).sum()), daily_target.loc[daily_target["is_valid_target_day"] & daily_target["historical_valid_day_count_45d"].ge(threshold), "household_id"].nunique())
    for threshold in sorted(set([config.target_min_history_days, *config.sensitivity_history_days]))
], columns=["minimum_valid_history_days", "eligible_target_days", "eligible_households"])

publish_table(target_eligibility_summary, "03_09_target_eligibility_summary.csv", "Target eligibility")
publish_table(ineligibility_summary, "03_10_target_ineligibility_reasons.csv", "Target ineligibility reasons")
publish_table(prevalence_p95, "03_11_target_prevalence_p95.csv", "P95 target prevalence")
publish_table(prevalence_p90, "03_12_target_prevalence_p90.csv", "P90 target prevalence")
publish_table(target_by_household, "03_13_target_prevalence_by_household.csv", "Target prevalence by household", display_rows=20)
publish_table(target_by_month, "03_14_target_prevalence_by_month.csv", "Target prevalence by month")
publish_table(history_sensitivity, "03_15_history_threshold_sensitivity.csv", "History eligibility sensitivity")

daily_target = daily_target.drop(columns="target_month")
write_csv_atomic(daily_target, TARGET_OUTPUT_PATH)
target_dtype_contract = write_dtype_contract(daily_target, TARGET_OUTPUT_PATH, TARGET_DTYPE_CONTRACT_PATH)
target_signature = csv_signature(TARGET_OUTPUT_PATH, len(daily_target), len(daily_target.columns))

target_missingness_after = pd.Series(0, index=daily_target.columns, dtype="int64")
target_roundtrip_rows = 0
for roundtrip_chunk in read_dataset_csv(TARGET_OUTPUT_PATH, target_dtype_contract, chunksize=100_000):
    target_roundtrip_rows += len(roundtrip_chunk)
    target_missingness_after = target_missingness_after.add(roundtrip_chunk.isna().sum(), fill_value=0).astype("int64")

target_roundtrip_columns = [
    "household_id", "measurement_date", "target_day_peak_kw", "historical_valid_day_count_45d",
    "historical_peak_p95_45d", "historical_peak_p90_45d", "target_eligible",
    "target_high_demand_p95", "target_high_demand_p90",
]
target_roundtrip = read_dataset_csv(TARGET_OUTPUT_PATH, target_dtype_contract, usecols=target_roundtrip_columns)
eligible_roundtrip = target_roundtrip["target_eligible"].fillna(False).astype(bool)
recomputed_p95 = target_roundtrip.loc[eligible_roundtrip, "target_day_peak_kw"].gt(target_roundtrip.loc[eligible_roundtrip, "historical_peak_p95_45d"]).astype("int8")
recomputed_p90 = target_roundtrip.loc[eligible_roundtrip, "target_day_peak_kw"].gt(target_roundtrip.loc[eligible_roundtrip, "historical_peak_p90_45d"]).astype("int8")
target_csv_checks = pd.DataFrame([
    ("target_csv_row_count", target_roundtrip_rows == len(daily_target) == len(daily), f"{target_roundtrip_rows:,}"),
    ("target_csv_households", target_roundtrip["household_id"].nunique() == daily["household_id"].nunique(), str(target_roundtrip["household_id"].nunique())),
    ("target_csv_keys_unique", not target_roundtrip.duplicated(["household_id", "measurement_date"]).any(), str(int(target_roundtrip.duplicated(["household_id", "measurement_date"]).sum()))),
    ("target_csv_missingness_matches", target_missingness_after.equals(daily_target.isna().sum().astype("int64")), str((target_missingness_after - daily_target.isna().sum()).abs().sum())),
    ("target_csv_eligible_rows", int(eligible_roundtrip.sum()) == int(daily_target["target_eligible"].sum()), str(int(eligible_roundtrip.sum()))),
    ("target_p95_recomputes_after_csv", recomputed_p95.reset_index(drop=True).equals(target_roundtrip.loc[eligible_roundtrip, "target_high_demand_p95"].astype("int8").reset_index(drop=True)), str(int(recomputed_p95.sum()))),
    ("target_p90_recomputes_after_csv", recomputed_p90.reset_index(drop=True).equals(target_roundtrip.loc[eligible_roundtrip, "target_high_demand_p90"].astype("int8").reset_index(drop=True)), str(int(recomputed_p90.sum()))),
    ("ineligible_targets_missing_after_csv", target_roundtrip.loc[~eligible_roundtrip, ["target_high_demand_p95", "target_high_demand_p90"]].isna().all().all(), "nullable targets retained"),
    ("threshold_precision_preserved", np.allclose(target_roundtrip["historical_peak_p95_45d"], daily_target["historical_peak_p95_45d"], equal_nan=True, rtol=1e-15, atol=1e-15) and np.allclose(target_roundtrip["historical_peak_p90_45d"], daily_target["historical_peak_p90_45d"], equal_nan=True, rtol=1e-15, atol=1e-15), "P95/P90 thresholds"),
    ("target_dtype_contract_written", TARGET_DTYPE_CONTRACT_PATH.exists(), str(TARGET_DTYPE_CONTRACT_PATH)),
], columns=["check", "passed", "evidence"])
publish_table(target_csv_checks, "03_03_target_csv_roundtrip_validation.csv", "Target CSV round-trip validation")
if not target_csv_checks["passed"].all():
    raise RuntimeError(f"Target CSV round-trip validation failed: {target_csv_checks.loc[~target_csv_checks.passed, 'check'].tolist()}")
print(f"Saved daily target dataset: {TARGET_OUTPUT_PATH} ({len(daily_target):,} rows)")
STAGE_3_COMPLETE = True

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
eligible_history = daily_target.loc[daily_target["is_valid_target_day"], "historical_valid_day_count_45d"]
axes[0, 0].hist(eligible_history, bins=np.arange(0, config.target_history_window_days + 2), color="#4C78A8")
axes[0, 0].axvline(config.target_min_history_days, color="#E45756", linestyle="--", label=f"Primary minimum = {config.target_min_history_days}")
axes[0, 0].axvline(config.sensitivity_history_days[1], color="#F58518", linestyle="--", label=f"Sensitivity = {config.sensitivity_history_days[1]}")
axes[0, 0].set_title("Qualified history days in previous 45 days")
axes[0, 0].legend()

threshold_values = daily_target.loc[daily_target["target_eligible"], "historical_peak_p95_45d"]
axes[0, 1].hist(np.log1p(threshold_values), bins=70, color="#54A24B")
axes[0, 1].set_title("Rolling household P95 thresholds")
axes[0, 1].set_xlabel("log(1 + P95 kW)")

prevalence_plot = pd.concat([prevalence_p95, prevalence_p90], ignore_index=True)
sns.barplot(data=prevalence_plot, x="target_definition", y="positive_event_rate", color="#B279A2", ax=axes[0, 2])
axes[0, 2].set_title("P90 versus P95 event prevalence")
axes[0, 2].tick_params(axis="x", rotation=20)
axes[0, 2].set_xlabel("")

sns.lineplot(data=target_by_month, x="target_month", y="p95_event_rate", marker="o", color="#E45756", ax=axes[1, 0])
axes[1, 0].set_title("P95 event rate over time")
axes[1, 0].tick_params(axis="x", rotation=45)
axes[1, 0].set_xlabel("Month")

axes[1, 1].hist(target_by_household["p95_event_rate"], bins=50, color="#72B7B2")
axes[1, 1].set_title("Household P95 event-rate distribution")
axes[1, 1].set_xlabel("P95 event rate")

scatter_data = daily_target.loc[daily_target["target_eligible"], ["target_day_peak_kw", "historical_peak_p95_45d", "target_high_demand_p95"]]
if len(scatter_data) > 30_000:
    scatter_data = scatter_data.sample(30_000, random_state=config.random_state)
sns.scatterplot(data=scatter_data, x="historical_peak_p95_45d", y="target_day_peak_kw", hue="target_high_demand_p95", alpha=0.35, s=18, ax=axes[1, 2], palette={0: "#4C78A8", 1: "#E45756"})
limit = max(scatter_data["target_day_peak_kw"].quantile(0.995), scatter_data["historical_peak_p95_45d"].quantile(0.995))
axes[1, 2].plot([0, limit], [0, limit], color="black", linestyle="--", linewidth=1)
axes[1, 2].set_xlim(0, limit)
axes[1, 2].set_ylim(0, limit)
axes[1, 2].set_title("Target-day peak versus prior P95")
fig.suptitle("Leakage-free household-relative target audit", fontsize=15)
fig.tight_layout()
publish_figure(fig, "03_02_target_construction_dashboard.png")

## Final validation and reproducibility manifest

In [ ]:
input_signature_after = input_signature(INTERVAL_PATH, preprocessing_manifest)

expected_tables = [
    "03_00_stage_0_validation.csv", "03_01_interval_input_validation.csv", "03_01_stage_1_validation.csv",
    "03_02_required_column_validation.csv", "03_02_stage_2_validation.csv", "03_02_csv_roundtrip_validation.csv",
    "03_03_aggregation_eligibility_summary.csv", "03_03_stage_3_validation.csv", "03_03_target_csv_roundtrip_validation.csv",
    "03_04_household_day_population_summary.csv", "03_05_daily_completeness_summary.csv",
    "03_05a_exact_interval_coverage_distribution.csv",
    "03_06_full_missing_day_summary.csv", "03_07_daily_metric_summary.csv", "03_08_daily_network_metadata_summary.csv",
    "03_09_target_eligibility_summary.csv", "03_10_target_ineligibility_reasons.csv",
    "03_11_target_prevalence_p95.csv", "03_12_target_prevalence_p90.csv",
    "03_13_target_prevalence_by_household.csv", "03_14_target_prevalence_by_month.csv",
    "03_15_history_threshold_sensitivity.csv",
]
expected_figures = ["03_01_household_day_construction_dashboard.png", "03_02_target_construction_dashboard.png"]
missing_tables = [name for name in expected_tables if not (TABLE_DIR / name).exists() or (TABLE_DIR / name).stat().st_size == 0]
missing_figures = [name for name in expected_figures if not (FIGURE_DIR / name).exists() or (FIGURE_DIR / name).stat().st_size == 0]
published_suffixes = sorted({path.suffix.lower() for path in PUBLISHED_TABLE_PATHS})

final_checks = pd.DataFrame([
    ("stage_0_complete", STAGE_0_COMPLETE, "configuration and input contracts"),
    ("stage_1_complete", STAGE_1_COMPLETE, "full interval CSV validation"),
    ("stage_2_complete", STAGE_2_COMPLETE, f"daily rows={daily_signature['rows']:,}"),
    ("stage_3_complete", STAGE_3_COMPLETE, f"target rows={target_signature['rows']:,}"),
    ("interval_input_unchanged", input_signature_after == input_signature_before, str(input_signature_after)),
    ("daily_and_target_row_counts_match", daily_signature["rows"] == target_signature["rows"] == len(daily_target) == len(daily), f"{len(daily_target):,}"),
    ("all_required_tables_exist", not missing_tables, str(missing_tables)),
    ("all_required_figures_exist", not missing_figures, str(missing_figures)),
    ("all_published_tables_are_csv", published_suffixes == [".csv"], str(published_suffixes)),
    ("daily_and_target_outputs_are_csv", OBSERVED_DAILY_OUTPUT_PATH.suffix == DAILY_OUTPUT_PATH.suffix == TARGET_OUTPUT_PATH.suffix == ".csv", f"{OBSERVED_DAILY_OUTPUT_PATH.name}; {DAILY_OUTPUT_PATH.name}; {TARGET_OUTPUT_PATH.name}"),
    ("dtype_contracts_exist", OBSERVED_DAILY_DTYPE_CONTRACT_PATH.exists() and DAILY_DTYPE_CONTRACT_PATH.exists() and TARGET_DTYPE_CONTRACT_PATH.exists(), f"{OBSERVED_DAILY_DTYPE_CONTRACT_PATH.name}; {DAILY_DTYPE_CONTRACT_PATH.name}; {TARGET_DTYPE_CONTRACT_PATH.name}"),
    ("observed_only_dataset_contains_observed_rows", OBSERVED_DAILY_OUTPUT_PATH.exists() and len(observed_daily) == int(daily["day_observed"].sum()), f"{len(observed_daily):,}"),
    ("no_interval_imputation", audit_counts["is_imputed"] == 0, str(audit_counts["is_imputed"])),
    ("no_outlier_removal", np.isclose(daily_target["observed_daily_peak_kw"].max(), 43.2648, atol=1e-9), str(daily_target["observed_daily_peak_kw"].max())),
    ("target_not_filled_for_ineligible_rows", bool(ineligible_targets_missing), "nullable target retained"),
    ("sampled_leakage_audit_passed", len(sampled_recomputation_errors) == 0, str(sampled_recomputation_errors[:10])),
    ("daily_csv_roundtrip_passed", bool(daily_csv_checks["passed"].all()), str(daily_csv_checks.loc[~daily_csv_checks.passed, "check"].tolist())),
    ("target_csv_roundtrip_passed", bool(target_csv_checks["passed"].all()), str(target_csv_checks.loc[~target_csv_checks.passed, "check"].tolist())),
], columns=["check", "passed", "evidence"])
publish_table(final_checks, "03_final_validation_checks.csv", "Final validation")
failed_final_checks = final_checks.loc[~final_checks["passed"], "check"].tolist()
if failed_final_checks:
    raise RuntimeError(f"Final validation failed: {failed_final_checks}")

run_manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "research_stage": "household_day_and_target",
    "completed_stages": [0, 1, 2, 3], "configuration": asdict(config), "input_signature": input_signature_after,
    "outputs": {
        "observed_daily_dataset": str(OBSERVED_DAILY_OUTPUT_PATH), "observed_daily_dtype_contract": str(OBSERVED_DAILY_DTYPE_CONTRACT_PATH),
        "observed_daily_signature": observed_daily_signature,
        "daily_dataset": str(DAILY_OUTPUT_PATH), "daily_dtype_contract": str(DAILY_DTYPE_CONTRACT_PATH),
        "daily_signature": daily_signature, "daily_rows": int(daily_signature["rows"]),
        "target_dataset": str(TARGET_OUTPUT_PATH), "target_dtype_contract": str(TARGET_DTYPE_CONTRACT_PATH),
        "target_signature": target_signature, "target_rows": int(target_signature["rows"]),
        "tables_directory": str(TABLE_DIR), "figures_directory": str(FIGURE_DIR),
    },
    "population": {
        "households": int(daily_target["household_id"].nunique()),
        "observed_household_days": int(daily_target["day_observed"].sum()),
        "continuous_calendar_rows": int(len(daily_target)),
        "full_missing_days": int(daily_target["is_full_missing_day"].sum()),
        "complete_days": int(daily_target["is_complete_day"].sum()),
        "target_eligible_rows": int(daily_target["target_eligible"].sum()),
        "target_eligible_households": int(daily_target.loc[daily_target["target_eligible"], "household_id"].nunique()),
    },
    "target_policy_status": selected_target_config["status"],
    "target_configuration_id": config.target_configuration_id,
    "target_policy_source": str(TARGET_CONFIG_PATH),
    "target_policy_decision_evidence": evidence,
    "target_policy_rationale": {
        "target_completeness": f"{config.target_day_min_intervals}/96; evidence is linked from the frozen 03A configuration",
        "history_completeness": f"{config.history_day_min_intervals}/96 valid historical days",
        "window_and_support": f"{config.target_history_window_days} calendar days with at least {config.target_min_history_days} valid days ({config.minimum_history_fraction:.0%} support)",
        "selection_basis": "target stability, history support, retention, and label agreement evidence; no downstream model performance"
    },
    "target_results": {
        "p95_positive_events": int(daily_target["target_high_demand_p95"].sum(skipna=True)),
        "p95_positive_rate": float(daily_target.loc[daily_target["target_eligible"], "target_high_demand_p95"].mean()),
        "p90_positive_events": int(daily_target["target_high_demand_p90"].sum(skipna=True)),
        "p90_positive_rate": float(daily_target.loc[daily_target["target_eligible"], "target_high_demand_p90"].mean()),
        "eligible_at_minimum_history_rows": int(eligible_minimum.sum()), "eligible_at_minimum_history_households": int(daily_target.loc[eligible_minimum, "household_id"].nunique()),
        "eligible_at_stricter_history_rows": int(eligible_sensitivity.sum()), "eligible_at_stricter_history_households": int(daily_target.loc[eligible_sensitivity, "household_id"].nunique()),
    },
    "target_audit_only_columns": ["target_day_peak_kw", "historical_peak_p95_45d", "historical_peak_p90_45d", "target_audit_peak_above_p95_kw", "target_audit_peak_over_p95_ratio"],
    "target_key_checksum": target_key_checksum,
    "validation": {row.check: bool(row.passed) for row in final_checks.itertuples(index=False)},
    "software": configuration_payload["software"],
}
write_json_atomic(run_manifest, RUN_MANIFEST_PATH)

print("All Stage 0–3 CSV validation checks passed.")
print(f"Daily dataset: {DAILY_OUTPUT_PATH}")
print(f"Target dataset: {TARGET_OUTPUT_PATH}")
print(f"Run manifest: {RUN_MANIFEST_PATH}")


## Next stage boundary

The frozen target policy identifier is carried on each output row. The next notebook may create past-only lag, rolling, relative, calendar, and event-history predictors from `03_daily_with_target.csv` and must use the reviewed 03B feature-quality evidence. It must use the recorded dtype contract, must not use target-day demand summaries or target-audit comparison fields as predictors, and must preserve chronological splitting with training-fitted imputation downstream.
